# 10. AL na wycinkach vs AL na obrazach (MoNuSeg, wybór losowy)

**Pytanie:** czy student uczy się szybciej, gdy oznaczamy **wycinki 256×256** zamiast całych obrazów
1000×1000, przy **tym samym budżecie pikseli**? (Uwaga promotora: cały obraz to setki jąder, warto
zobaczyć, jak rosną metryki, gdy budżet rośnie wycinkami.)

**Dwa przebiegi, ten sam przepis treningu** (zimny start z SAM ViT-L w każdej rundzie, maski wyroczni
`cpsam`, 200 epok × 64 fragmenty 256 px, batch 4, lr 5·10⁻⁵, wd 0,1, 3200 kroków na rundę):

| | AL na obrazach (`al/random_loop.py`) | AL na wycinkach (`al/tile_random_loop.py`) |
|---|---|---|
| pula | 30 obrazów treningowych (bez walidacji) | te same obrazy pocięte na 16 wycinków 256 px (krok 248 px, zakładka 8 px) = 480 |
| adnotacja | maska wyroczni całego obrazu | maska wyroczni liczona na całym obrazie, wycięta do wycinka (przecięte jądra jako fragmenty) |
| runda | +2 obrazy | +32 wycinki (= 2 obrazy pikselowo) |
| wybór | losowa permutacja obrazów (seed) | losowa permutacja wycinków (seed) |
| student widzi | całe obrazy | tylko oznaczone wycinki |

Dwa warianty AL na wycinkach (ten sam wybór i ta sama kolejność wycinków):
- **wycinki**: cel uczenia Cellpose (etykiety + flowy) liczony na wyciętej masce; jądro przecięte brzegiem
  jest fragmentem z własnym środkiem,
- **wycinki + kontekst** (`--full-image-flows --margin 64`): flowy liczone na pełnej masce wyroczni i wycięte
  (przecięte jądro zachowuje flow do prawdziwego środka), a student widzi dodatkowo 64 px tkanki wokół
  wycinka (384 px), **bez adnotacji**: strata pomija te piksele, więc budżet oznaczonych pikseli się nie
  zmienia. Kontekst zastępuje czarne dopełnienie, które powstaje przy losowym obrocie (±180°)
  i skalowaniu (0,75–1,25×) wycinka 256 px do fragmentu 256 px w augmentacji Cellpose.

Oś X wszędzie: **ekwiwalent obrazów** = liczba oznaczonych pikseli / 1000² (wycinki: n / 16).

**Ocena:**
- walidacja (7 obrazów, GT) jak dotąd: PQ, AJI na całych obrazach; tylko do monitorowania,
- **test** (14 obrazów, GT; `al/test_eval.py`, używany wyłącznie do raportu): student segmentuje całe
  obrazy testowe, metryki (a) per obraz, (b) per wycinek 256 px tej samej siatki: średnia po wycinkach
  z przycięcia obu masek (`tile_pq`, `tile_aji`, ...) i z przypisania jąder po środku (`tile_own_pq`).

**Zakres:** seed 0 obu przebiegów (AL na wycinkach na razie tylko seed 0). Dla walidacji AL na obrazach
pokazujemy też rozrzut 6 seedów, żeby ocenić, czy różnica wychodzi poza zmienność między seedami.

In [ ]:
SEED = 0
METRICS = ["pq", "aji", "dq", "pq@0.75"]  # dq = F1@0.5
TILE_METRICS = ["tile_pq", "tile_aji", "tile_dq", "tile_own_pq"]

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

from histopathology_pnw.config import Settings

OUT = Settings().outputs_dir
RUNS = {
    "obrazy": OUT / "al_random" / "full",
    "wycinki": OUT / "al_random_tiles" / "full",
    "wycinki + kontekst": OUT / "al_random_tiles" / "fullflows_m64",
}
TEST = {
    "obrazy": OUT / "test_eval" / "al_random_full",
    "wycinki": OUT / "test_eval" / "al_random_tiles_full",
    "wycinki + kontekst": OUT / "test_eval" / "al_random_tiles_fullflows_m64",
}
TILE_RUNS = [r for r in RUNS if r != "obrazy"]
for p in [*RUNS.values(), *TEST.values()]:
    if not p.is_dir():
        raise FileNotFoundError(f"{p} not found: run al/tile_random_loop.py and al/test_eval.py first")
COLORS = {"obrazy": "tab:blue", "wycinki": "tab:orange", "wycinki + kontekst": "tab:green"}


def equiv(df, run):
    return df["n_labeled"] / (1 if run == "obrazy" else 16)


val = {}
for run, root in RUNS.items():
    df = pd.read_csv(root / f"seed{SEED}" / "results.csv")
    val[run] = df.assign(run=run, x=equiv(df, run))
test = {}
for run, root in TEST.items():
    df = pd.read_csv(root / f"seed{SEED}" / "test_metrics.csv")
    test[run] = df.assign(run=run, x=equiv(df, run))
oracle_test = pd.read_csv(TEST["obrazy"] / "oracle.csv").iloc[0]
print({r: len(d) for r, d in test.items()}, "rund z wynikiem testu")

## 1. Co student dostaje w każdej rundzie

Przy tym samym budżecie pikseli wycinki pochodzą z wielu obrazów (większa różnorodność tkanek i barwień),
a liczba oznaczonych jąder jest podobna (różni się tylko tym, ile jąder mają wylosowane miejsca).

In [ ]:
masks_dir = OUT / "oracle_masks" / "monuseg"
n_nuc = {}


def nuclei_of_image(i):
    if i not in n_nuc:
        n_nuc[i] = int(tifffile.imread(masks_dir / f"{i}.tif").max())
    return n_nuc[i]


img = val["obrazy"].copy()
img["n_source_images"] = img["n_labeled"]
img["n_labeled_nuclei"] = [sum(nuclei_of_image(i) for i in json.loads(s)) for s in img["labeled_ids"]]
cols = ["x", "n_labeled", "n_source_images", "n_labeled_nuclei"]
supply = pd.concat([img[cols].assign(run="obrazy"), val["wycinki"][cols].assign(run="wycinki")])  # kontekst: ten sam wybór
supply.pivot(index="x", columns="run", values=["n_source_images", "n_labeled_nuclei"]).astype(int)

## 2. Walidacja (monitorowanie; seed 0 obu przebiegów + rozrzut 6 seedów AL na obrazach)

In [ ]:
seeds_img = sorted(int(p.name[4:]) for p in RUNS["obrazy"].glob("seed*") if (p / "results.csv").is_file())
band = pd.concat(
    [pd.read_csv(RUNS["obrazy"] / f"seed{s}" / "results.csv").assign(seed=s) for s in seeds_img]
)
print("seedy AL na obrazach z results.csv:", seeds_img)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, m in zip(axes, ["val_pq", "val_aji"]):
    if len(seeds_img) > 1:
        g = band.groupby("n_labeled")[m]
        ax.fill_between(g.min().index, g.min(), g.max(), color=COLORS["obrazy"], alpha=0.15,
                        label=f"obrazy, min–max {len(seeds_img)} seedów")
    for run, df in val.items():
        ax.plot(df["x"], df[m], "o-", color=COLORS[run], label=f"{run}, seed {SEED}")
    ax.set_xlabel("ekwiwalent obrazów (oznaczone piksele / 1000²)")
    ax.set_ylabel(m.replace("val_", "").upper() + " (val, całe obrazy)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()

## 3. Test: metryki per obraz i per wycinek

Linia przerywana: wyrocznia na teście względem GT (sufit; wyrocznia widziała MoNuSeg w treningu
Cellpose-SAM, więc to sufit „łatwy”).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
for ax, m in zip(axes.ravel(), METRICS + TILE_METRICS):
    for run, df in test.items():
        ax.plot(df["x"], df[m], "o-", color=COLORS[run], label=run)
    ax.axhline(oracle_test[m], color="k", ls="--", lw=1, label="wyrocznia")
    ax.set_title(("per obraz: " if m in METRICS else "per wycinek: ") + m)
    ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("ekwiwalent obrazów")
axes[0, 0].legend()
plt.tight_layout()

In [ ]:
cols = METRICS + TILE_METRICS
diffs = {}
for run in TILE_RUNS:
    both = pd.merge(
        test["obrazy"][["x", *cols]], test[run][["x", *cols]], on="x", suffixes=("_img", "_tile")
    )
    diffs[run] = pd.DataFrame({m: both[f"{m}_tile"] - both[f"{m}_img"] for m in cols}).set_index(both["x"])
print("różnica wariant − obrazy na teście (dodatnia = AL na wycinkach lepszy), seed 0")
pd.concat({r: d[["pq", "aji", "pq@0.75", "tile_pq"]] for r, d in diffs.items()}, axis=1).round(3)

In [ ]:
summary = pd.concat(
    {
        run: pd.DataFrame(
            {
                "średnia różnica": d.mean().round(3),
                "rundy z przewagą wycinków": (d > 0).sum().astype(str) + f"/{len(d)}",
                "do 8 ekw. obrazów": d[d.index <= 8].mean().round(3),
                "od 10 ekw. obrazów": d[d.index >= 10].mean().round(3),
            }
        )
        for run, d in diffs.items()
    }
)
summary

In [ ]:
def reach(df, m, frac=0.95):
    # najmniejszy budżet, przy którym student osiąga frac × PQ wyroczni na teście
    ok = df.loc[df[m] >= frac * oracle_test[m], "x"]
    return float(ok.min()) if len(ok) else np.nan


pd.DataFrame(
    {run: {f"{m}: {f:.0%} wyroczni": reach(df, m, f) for m in ["pq", "tile_pq"] for f in (0.9, 0.95)}
     for run, df in test.items()}
)

## 3b. Testy diagnostyczne: skąd strata wycinków (pojedyncze treningi, seed 0)

Te same oznaczone wycinki, zmienia się tylko cel uczenia / to, co student widzi:
- *flowy z pełnej maski* (`al_random_tiles/fullflows`, rundy 160 i 480): flowy liczone na pełnej masce
  wyroczni i wycięte, bez kontekstu,
- *wycinki + kontekst*: jak wyżej + 64 px tkanki bez adnotacji.

In [ ]:
diag = {
    "obrazy": ("al_random_full", {160: 10, 480: 30}),
    "wycinki": ("al_random_tiles_full", {160: 160, 480: 480}),
    "flowy z pełnej maski": ("al_random_tiles_fullflows", {160: 160, 480: 480}),
    "wycinki + kontekst": ("al_random_tiles_fullflows_m64", {160: 160, 480: 480}),
}
rows = []
for name, (folder, rounds) in diag.items():
    path = OUT / "test_eval" / folder / f"seed{SEED}" / "test_metrics.csv"
    if not path.is_file():
        continue
    t = pd.read_csv(path).set_index("n_labeled")
    for budget, n in rounds.items():
        if n in t.index:
            rows.append({"budżet (wycinki)": budget, "student": name,
                         **t.loc[n, ["pq", "sq", "dq", "pq@0.75", "aji", "tile_pq"]].to_dict()})
pd.DataFrame(rows).set_index(["budżet (wycinki)", "student"]).round(4)

## 4. Wnioski (seed 0; przebiegi z 22–23.11.2026)

1. **Zwykłe wycinki przegrywają z obrazami w 14 z 15 rund** (test PQ średnio −0,023, PQ@0,75 −0,05).
   Wygrywają tylko przy 2 ekw. obrazach (różnorodność: 23 obrazy źródłowe zamiast 2). Krzywa jest płaska
   (~0,58), a przy 480 wycinkach, czyli **tych samych pikselach co 30 obrazów**, student nadal przegrywa
   (0,576 vs 0,591): strata wynika ze sposobu uczenia na wycinkach, nie z wyboru.
2. **Przyczyną jest głównie brak tkanki wokół wycinka, nie fragmenty jąder** (p. 3b):
   - same flowy z pełnej maski nic nie dają (runda 160: 0,584 vs 0,583),
   - 64 px kontekstu bez adnotacji (budżet bez zmian) daje +0,010 PQ i +0,02–0,03 PQ@0,75. Augmentacja
     Cellpose obraca fragment o ±180° i skaluje 0,75–1,25×, więc z wycinka 256 px część fragmentu
     to czarne dopełnienie; kontekst zastępuje je tkanką.
3. **Wycinki + kontekst są lepsze od zwykłych wycinków w każdej z 15 rund** na teście i prawie w każdej
   na walidacji. Różnica do AL na obrazach spada o połowę: test PQ średnio −0,012 (było −0,023), PQ@0,75
   −0,024 (było −0,049). Do 8 ekw. obrazów to praktycznie remis (−0,003; przy 2 ekw. +0,016), od 20 ekw.
   obrazów też niemal remis (0,000 do −0,014; przy 30: +0,002). Na walidacji krzywa mieści się w paśmie
   6 seedów AL na obrazach.
4. **Pozostała różnica jest w środku zakresu (10–18 ekw. obrazów, −0,02 do −0,03)**, gdzie krzywa testowa
   AL na obrazach (seed 0) ma garb do 0,622. Na walidacji tego garbu prawie nie ma, więc część różnicy
   może być szczęściem tego seeda; bez kolejnych seedów nie da się tego rozstrzygnąć.
5. **Krzywe wycinków są płaskie od ok. 4 ekw. obrazów** (~0,59 z kontekstem): więcej losowych wycinków
   prawie nie pomaga. Dla AL to dobra wiadomość: budżet, przy którym student osiąga 90% PQ wyroczni,
   to 2 ekw. obrazy z kontekstem wobec 4 obrazów.
6. Metryki per wycinek (z przycięcia i po środku jądra) dają te same wnioski co per obraz.

**Ograniczenia:** jeden seed obu metod, 14 obrazów testowych, pojedynczy trening na punkt
(wahania między sąsiednimi rundami ok. 0,01 PQ).

**Co dalej:** (a) AL na wycinkach uczyć domyślnie z kontekstem (`--full-image-flows --margin 64`);
(b) seedy 1–2 dla wycinków z kontekstem i test dla seedów 1–2 AL na obrazach (checkpointy są na rcmc), żeby
sprawdzić p. 4; (c) dopiero potem selektory wycinków (rank_kd, podgląd wyroczni) na tym samym przepisie.